# Attribution precision

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# interpretation results (rows=snps, cols=attribution methods)
model_dir = "./experiments/dnn_decoy"
attrs_path = os.path.join(model_dir, "attributions.csv")
attrs_df = pd.read_csv(attrs_path).set_index('snp')
algo_names = list(attrs_df.columns)
print(f"Attributions file: {attrs_path}")
print(f"Algorithms: {algo_names}")

In [ ]:
def ppv_decoy(scores, quantile):
    quantile_thresh = scores.quantile(quantile)
    n_topk = len(scores[scores > quantile_thresh])
    ranks = np.argsort(scores.values)[::-1]
    topk_indices = ranks[:n_topk]
    topk_real = topk_indices[topk_indices < len(scores)//2]
    topk_decoy = topk_indices[topk_indices >= len(scores)//2]
    assert len(topk_real) + len(topk_decoy) == n_topk
    ppv = 1 - (len(topk_decoy) / len(topk_real)) if len(topk_real) > 0 else 0
    return ppv

In [ ]:
n_scores = len(attrs_df)

# quantiles to evaluate
quantiles = [0.80, 0.90, 0.95, 0.97, .98, .99]

# alternative approach to generate eval quantiles
#q_start, q_end, q_step = 0.80, 1.0, 0.01
#quantile_list = np.arange(q_start, q_end, q_step)

In [ ]:
results = []
for q in quantiles:
    k = int(np.round((1-q)*n_scores))
    percentile = 100 * q
    results_q = [
        {'algo': algo, 'quantile': q, 
         'percentile': percentile, 'k': k, 
         'ppv': ppv_decoy(attrs_df.loc[:, algo], q)} 
         for algo in algo_names
    ]
    results.extend(results_q)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# save results
ppv_results_file = os.path.join(model_dir, 'ppv_results.csv')
results_df.to_csv(ppv_results_file, index=False)
print(f"PPV results saved to: {ppv_results_file}")

In [ ]:
sg_algos = [col for col in attrs_df.columns \
            if col.endswith('SG')]
nosg_algos = [col for col in attrs_df.columns \
              if not col.endswith('SG')]
sg_results_df = results_df[results_df['algo'].isin(sg_algos)]
nosg_results_df = results_df[results_df['algo'].isin(nosg_algos)]

In [ ]:
# Plot mean ppv at each quantile for 
# SmoothGrad vs non-SmoothGrad (shading is ± 1 stdev)
fig, ax = plt.subplots(figsize=(6.4, 4.8), dpi=300)
mean_sg_ppv = sg_results_df.groupby('q')['ppv'].mean()
std_sg_ppv = sg_results_df.groupby('q')['ppv'].std()
mean_nosg_ppv = nosg_results_df.groupby('q')['ppv'].mean()
std_nosg_ppv = nosg_results_df.groupby('q')['ppv'].std()
ax.plot(
    mean_nosg_ppv.index, 
    mean_nosg_ppv.values, 
    label='without SmoothGrad', 
    linewidth=1.5, color='tab:blue', 
)
ax.fill_between(
    mean_nosg_ppv.index, 
    mean_nosg_ppv.values - std_nosg_ppv.values, 
    mean_nosg_ppv.values + std_nosg_ppv.values, 
    color='tab:blue', alpha=0.3, 
)
ax.plot(
    mean_sg_ppv.index, 
    mean_sg_ppv.values, 
    label='with SmoothGrad', 
    linewidth=1.5, color='tab:orange', 
)
ax.fill_between(
    mean_sg_ppv.index, 
    mean_sg_ppv.values - std_sg_ppv.values, 
    mean_sg_ppv.values + std_sg_ppv.values, 
    color='tab:orange', alpha=0.3,
)
ax.set_title('Attribution Precision')
ax.set_xlabel('quantile')
ax.set_ylabel('precision')
ax.set_xlim(np.min(results_df['q']), np.max(results_df['q']))
plot_quantiles = np.round(sorted(results_df['q']), 2)
plot_quantiles_even = [float(q) for q in plot_quantiles if int(q*100) % 2 == 0]
x_ticks = np.round(np.arange(0.8, 0.99, 0.02), 2)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_ticks,)
y_ticks = np.round(np.arange(0.1, 1.1, 0.1), 1)
ax.set_yticks(y_ticks)
ax.set_yticklabels(y_ticks)
ax.grid(True, alpha=0.5, linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(fontsize='small')
plt.tight_layout()
ppv_plot_fn = os.path.join(model_dir, 'ppv_sg_vs_nosg_plot.tiff')
plt.savefig(
    ppv_plot_fn, dpi=300,
    bbox_inches='tight', format='tiff',
    pil_kwargs={"compression": "tiff_lzw"}
)
print(f"Plot saved to:\n{ppv_plot_fn}")
plt.show()